In [37]:
# ============================================================
# 🔗 Direct PostgreSQL Connection (No dependencies on modules)
# ============================================================
from sqlalchemy import create_engine, text
import pandas as pd

# --- Direct connection parameters ---
user = "postgres"
password = "tip_pwd"
host = "localhost"
port = "5432"
database = "tip"  # unified database with schemas raw/silver/gold

# --- Build connection URL ---
url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"

# --- Create engine ---
engine = create_engine(url, pool_pre_ping=True, future=True)

# --- Query: get the most recent 10 CVEs ---
query = """
SELECT *
FROM raw.cve_details;
"""

# --- Execute and load to pandas ---
with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)

In [11]:
df.shape

(313519, 9)

In [38]:
df.head()

,cve_id,published_date,last_modified,remotely_exploit,source_identifier,category,affected_products,cvss_scores,loaded_at
0,CVE-1999-0095,1988-10-01T04:00:00.000,2025-04-03T01:03:51.193,None,cve@mitre.org,,"[{'vendor': 'eric_allman', 'product': 'sendmai...","[{'type': 'Primary', 'score': 10.0, 'vector': ...",2025-10-21 23:23:39.502088+00:00
1,CVE-1999-0082,1988-11-11T05:00:00.000,2025-04-03T01:03:51.193,None,cve@mitre.org,,"[{'vendor': 'ftp', 'product': 'ftp'}, {'vendor...","[{'type': 'Primary', 'score': 10.0, 'vector': ...",2025-10-21 23:23:39.502088+00:00
2,CVE-1999-1471,1989-01-01T05:00:00.000,2025-04-03T01:03:51.193,None,cve@mitre.org,,"[{'vendor': 'bsd', 'product': 'bsd'}]","[{'type': 'Primary', 'score': 7.2, 'vector': '...",2025-10-21 23:23:39.502088+00:00
3,CVE-1999-1122,1989-07-26T04:00:00.000,2025-04-03T01:03:51.193,None,cve@mitre.org,,"[{'vendor': 'sun', 'product': 'sunos'}]","[{'type': 'Primary', 'score': 4.6, 'vector': '...",2025-10-21 23:23:39.502088+00:00
4,CVE-1999-1467,1989-10-26T04:00:00.000,2025-04-03T01:03:51.193,None,cve@mitre.org,,"[{'vendor': 'sun', 'product': 'sunos'}]","[{'type': 'Primary', 'score': 10.0, 'vector': ...",2025-10-21 23:23:39.502088+00:00


In [40]:
pd.set_option('display.max_colwidth', None)
print(df["cvss_scores"].head())

0    [{'type': 'Primary', 'score': 10.0, 'vector': 'AV:N/AC:L/Au:N/C:C/I:C/A:C', 'version': '2.0', 'severity': 'HIGH', 'impact_score': 10.0, 'source_identifier': 'nvd@nist.gov', 'exploitability_score': 10.0}]
1    [{'type': 'Primary', 'score': 10.0, 'vector': 'AV:N/AC:L/Au:N/C:C/I:C/A:C', 'version': '2.0', 'severity': 'HIGH', 'impact_score': 10.0, 'source_identifier': 'nvd@nist.gov', 'exploitability_score': 10.0}]
2      [{'type': 'Primary', 'score': 7.2, 'vector': 'AV:L/AC:L/Au:N/C:C/I:C/A:C', 'version': '2.0', 'severity': 'HIGH', 'impact_score': 10.0, 'source_identifier': 'nvd@nist.gov', 'exploitability_score': 3.9}]
3     [{'type': 'Primary', 'score': 4.6, 'vector': 'AV:L/AC:L/Au:N/C:P/I:P/A:P', 'version': '2.0', 'severity': 'MEDIUM', 'impact_score': 6.4, 'source_identifier': 'nvd@nist.gov', 'exploitability_score': 3.9}]
4    [{'type': 'Primary', 'score': 10.0, 'vector': 'AV:N/AC:L/Au:N/C:C/I:C/A:C', 'version': '2.0', 'severity': 'HIGH', 'impact_score': 10.0, 'source_identifier': 'n

In [24]:
# ============================================================
# 🔗 Load Silver Table (like cve_cleaned.head())
# ============================================================
from sqlalchemy import create_engine, text
import pandas as pd

# --- Connection info ---
user = "postgres"
password = "tip_pwd"
host = "localhost"
port = "5432"
database = "tip"

# --- Build connection URL ---
url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"

# --- Create engine ---
engine = create_engine(url, pool_pre_ping=True, future=True)

# --- Query: select data from silver.cve_cleaned ---
query = """
SELECT *
FROM silver.cve_cleaned
ORDER BY published_date DESC NULLS LAST
LIMIT 10;
"""

# --- Execute and load to pandas ---
with engine.connect() as conn:
    cve_cleaned = pd.read_sql(text(query), conn)

# --- Show preview ---


In [25]:
cve_cleaned.head()

,cve_id,vulnarbilit,published_date,last_modified,loaded_at,remotely_exploit,source_identifier,affected_products,cvss_scores,created_at,updated_at
0,CVE-2025-61871,uncategorized,2025-10-10 05:15:33.587,2025-10-10 05:15:33.587,2025-10-21 23:25:03.061024,None,vultures@jpcert.or.jp,[],"[{""type"": ""Secondary"", ""score"": 8.4, ""vector"":...",2025-10-22 00:03:05.268825,2025-10-22 00:03:05.268825
1,CVE-2025-11570,xss,2025-10-10 05:15:33.380,2025-10-10 05:15:33.380,2025-10-21 23:25:03.061024,None,report@snyk.io,[],"[{""type"": ""Secondary"", ""score"": 4.8, ""vector"":...",2025-10-22 00:03:05.268825,2025-10-22 00:03:05.268825
2,CVE-2025-11569,path_traversal,2025-10-10 05:15:32.190,2025-10-10 05:15:32.190,2025-10-21 23:25:03.061024,None,report@snyk.io,[],"[{""type"": ""Secondary"", ""score"": 7.7, ""vector"":...",2025-10-22 00:03:05.268825,2025-10-22 00:03:05.268825
3,CVE-2025-11450,xss,2025-10-10 02:15:38.610,2025-10-10 02:15:38.610,2025-10-21 23:25:03.061024,None,psirt@servicenow.com,[],"[{""type"": ""Secondary"", ""score"": 5.3, ""vector"":...",2025-10-22 00:03:05.268825,2025-10-22 00:03:05.268825
4,CVE-2025-11449,xss,2025-10-10 02:15:38.440,2025-10-10 02:15:38.440,2025-10-21 23:25:03.061024,None,psirt@servicenow.com,[],"[{""type"": ""Secondary"", ""score"": 5.3, ""vector"":...",2025-10-22 00:03:05.268825,2025-10-22 00:03:05.268825


In [34]:
# ============================================================
# 🏆 Load all GOLD tables into pandas DataFrames
# Then you can call: cvss_v2.head(), dim_vendor.head(), etc.
# ============================================================
from sqlalchemy import create_engine, text
import pandas as pd

# --- Connection info ---
user = "postgres"
password = "tip_pwd"
host = "localhost"
port = "5432"
database = "tip"

# --- Create engine ---
url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(url, pool_pre_ping=True, future=True)

# --- List of tables in Gold layer ---
gold_tables = [
    "dim_cve",
    "dim_cvss_source",
    "dim_vendor",
    "dim_products",
    "cvss_v2",
    "cvss_v3",
    "cvss_v4",
    "bridge_cve_products"
]

# --- Load each table into a DataFrame variable ---
globals().update({
    table: pd.read_sql(text(f"SELECT * FROM gold.{table}"), engine)
    for table in gold_tables
})

# --- Confirmation ---
print("✅ All Gold tables loaded successfully!\n")

✅ All Gold tables loaded successfully!



In [35]:
cvss_v3.head()


,cvss_v3_id,cve_id,source_id,cvss_version,cvss_score,cvss_severity,cvss_vector,cvss_v3_base_av,cvss_v3_base_ac,cvss_v3_base_pr,cvss_v3_base_ui,cvss_v3_base_s,cvss_v3_base_c,cvss_v3_base_i,cvss_v3_base_a,cvss_exploitability_score,cvss_impact_score,created_at


In [36]:
# --- Now you can simply do, for example: ---
# dim_cve.head()
# cvss_v3.head()
# dim_vendor.head()
# bridge_cve_products.head()

,cve_id,vulnarbilit,published_date,last_modified,loaded_at,cve_year,remotely_exploit,source_identifier,created_at
0,CVE-1999-0001,input_validation,1999-12-30 05:00:00,2025-04-03 01:03:51.193,2025-10-21 23:23:39.502088,1999,None,cve@mitre.org,2025-10-22 00:10:26.591587
1,CVE-1999-0002,memory_corruption,1998-10-12 04:00:00,2025-04-03 01:03:51.193,2025-10-21 23:23:39.502088,1998,None,cve@mitre.org,2025-10-22 00:10:26.591587
2,CVE-1999-0003,uncategorized,1998-04-01 05:00:00,2025-04-03 01:03:51.193,2025-10-21 23:23:39.502088,1998,None,cve@mitre.org,2025-10-22 00:10:26.591587
3,CVE-1999-0004,uncategorized,1997-12-16 05:00:00,2025-04-03 01:03:51.193,2025-10-21 23:23:39.502088,1997,None,cve@mitre.org,2025-10-22 00:10:26.591587
4,CVE-1999-0005,uncategorized,1998-07-20 04:00:00,2025-04-03 01:03:51.193,2025-10-21 23:23:39.502088,1998,None,cve@mitre.org,2025-10-22 00:10:26.591587
